# Neuroglancer Annotation Pipeline

This notebook provides a workflow to:
1.  Generate a Neuroglancer link for a specific neuron.
2.  Annotate points in 3D (Spines vs. Non-Spines).
3.  Ingest the annotations back into Python.
4.  **Snap** these 3D points to the nearest mesh vertex.
5.  **Map** the vertices to HKS Segment IDs.
6.  **Export** a Machine Learning dataset (X=HKS features, y=Labels).

In [1]:
!pip install nglui
import numpy as np
import pandas as pd
from caveclient import CAVEclient
from meshparty import trimesh_io
from meshmash import condensed_hks_pipeline
import annotation_utils # Our new helper module

# --- CONFIGURATION ---
dataset_name = 'minnie65_public'
materialization = 1300
CACHE_DIR = 'meshes'
target_id = 864691135724233643 # Example Neuron ID

# Setup Client
client = CAVEclient(dataset_name)
client.version = materialization

mm = trimesh_io.MeshMeta(
    cv_path=client.info.segmentation_source(),
    disk_cache_path=CACHE_DIR
)


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: C:\Users\bkrou\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable


## Step 1: Generate Neuroglancer Link
Click the link below to open Neuroglancer. Use the 'annotations' layer to add points:
-   **Spines**: Add points to spines you want to label.
-   **Control**: You can optionally add points to dendrites/somas for negative labels.

In [2]:
# link = annotation_utils.generate_ng_link(target_id, client)
# try:
#     from IPython.display import display, HTML
#     display(HTML(f'<a href="{link}" target="_blank">OPEN NEUROGLANCER</a>'))
# except:
#     print(link)

# import caveclient
# from nglui import statebuilder

# client = caveclient.CAVEclient('minnie65_public')

# # Get a root id of a specific neuron
# root_id = client.materialize.query_table(
#     'nucleus_detection_v0',
#     filter_equal_dict={'id': target_id}
# )['pt_root_id']

# statebuilder.helpers.make_neuron_neuroglancer_link(
#     client,
#     root_id,
#     show_inputs=True,
#     show_outputs=True,
# )

import importlib
import annotation_utils
importlib.reload(annotation_utils)
from caveclient import CAVEclient

# 1. Setup client
client = CAVEclient('minnie65_public')

# 2. Your target neuron
target_id = 864691135724233643

# 3. Generate the link — this uses the OFFICIAL nglui helper
link = annotation_utils.generate_ng_link(target_id, client)

print("CLICK TO OPEN NEUROGLANCER:")
print(link)

CLICK TO OPEN NEUROGLANCER:
https://spelunker.cave-explorer.org/#!%7B%22position%22:%5B218809.0,161359.0,13929.0%5D,%22layout%22:%22xy-3d%22,%22dimensions%22:%7B%22x%22:%5B4e-09,%22m%22%5D,%22y%22:%5B4e-09,%22m%22%5D,%22z%22:%5B4e-08,%22m%22%5D%7D,%22crossSectionScale%22:1.0,%22projectionScale%22:50000.0,%22showSlices%22:false,%22layers%22:%5B%7B%22type%22:%22image%22,%22source%22:%5B%7B%22url%22:%22precomputed://https://bossdb-open-data.s3.amazonaws.com/iarpa_microns/minnie/minnie65/em%22,%22transform%22:%7B%22outputDimensions%22:%7B%22x%22:%5B4e-09,%22m%22%5D,%22y%22:%5B4e-09,%22m%22%5D,%22z%22:%5B4e-08,%22m%22%5D%7D%7D,%22subsources%22:%7B%7D,%22enableDefaultSubsources%22:true%7D%5D,%22name%22:%22imagery%22%7D,%7B%22type%22:%22segmentation%22,%22source%22:%5B%7B%22url%22:%22precomputed://https://storage.googleapis.com/iarpa_microns/minnie/minnie65/seg_m1300%22,%22transform%22:%7B%22outputDimensions%22:%7B%22x%22:%5B4e-09,%22m%22%5D,%22y%22:%5B4e-09,%22m%22%5D,%22z%22:%5B4e-08,%22m%2

In [3]:
import os
from caveclient import CAVEclient
from nglui import parser

client = CAVEclient('minnie65_public')

state_id = 5560000195854336
state_json = client.state.get_state_json(state_id)
state = parser.StateParser(state_json)

state.layer_dataframe()

,layer,type,source,archived
0,img,image,precomputed://https://bossdb-open-data.s3.amaz...,False
1,seg,segmentation_with_graph,graphene://https://minnie.microns-daf.com/segm...,False
2,syns_in,annotation,None,False
3,syns_out,annotation,None,False


In [4]:
state.annotation_dataframe()

,layer,anno_type,point,pointB,linked_segmentation,tags,anno_id,group_id,description
0,syns_in,point,"[294095, 196476, 24560]",NaN,[864691136333760691],[],b832a0b7fe04a34195fef0fbc6743260e389e68e,None,None
1,syns_in,point,"[294879, 196374, 24391]",NaN,[864691136333760691],[],09ac4a97322131421d292d96a13ecbeb52dea50a,None,None
2,syns_in,point,"[300246, 200562, 24297]",NaN,[864691136333760691],[],a9f1c484bbe037d3571525748455c19e6d792652,None,None
3,syns_in,point,"[300894, 201844, 24377]",NaN,[864691136333760691],[],2efe4f45ad2b1f8e11da480a205a907ae5224c21,None,None
4,syns_in,point,"[294742, 199552, 23392]",NaN,[864691136333760691],[],00661afd142462468d74167c2857c1d3f64b6870,None,None
...,...,...,...,...,...,...,...,...,...
5272,syns_out,point,"[277152, 200746, 22723]",NaN,[864691132294257136],[],92fce4b77aa345ad0c5c9e7c2d568b5de9aaa274,None,None
5273,syns_out,point,"[298884, 189782, 21453]",NaN,[864691132135519710],[],7aafe74e6a4be5daf64c4f21439ed1788faf0ef0,None,None
5274,syns_out,point,"[330182, 198986, 23862]",NaN,[864691132100215248],[],7f250c09e29df2a498236b9177292c0c68ac36f1,None,None
5275,syns_out,point,"[326552, 186446, 24792]",NaN,[864691131892380409],[],4199c5a222e64a13976d6628d91eea4807a4fd90,None,None


In [5]:
parser.tag_dictionary(state_json, layer_name='syns_out')


{1: 'targets_spine', 2: 'targets_shaft', 3: 'targets_soma'}

In [6]:
state.annotation_dataframe(expand_tags=True, split_points=True, point_resolution=[1,1,1])


,layer,anno_type,point,linked_segmentation,tags,anno_id,group_id,description,point_x,point_y,point_z,pointB_x,pointB_y,pointB_z,targets_spine,targets_shaft,targets_soma
0,syns_in,point,NaN,[864691136333760691],[],b832a0b7fe04a34195fef0fbc6743260e389e68e,None,None,1176380.0,785904.0,982400.0,NaN,NaN,NaN,NaN,NaN,NaN
1,syns_in,point,NaN,[864691136333760691],[],09ac4a97322131421d292d96a13ecbeb52dea50a,None,None,1179516.0,785496.0,975640.0,NaN,NaN,NaN,NaN,NaN,NaN
2,syns_in,point,NaN,[864691136333760691],[],a9f1c484bbe037d3571525748455c19e6d792652,None,None,1200984.0,802248.0,971880.0,NaN,NaN,NaN,NaN,NaN,NaN
3,syns_in,point,NaN,[864691136333760691],[],2efe4f45ad2b1f8e11da480a205a907ae5224c21,None,None,1203576.0,807376.0,975080.0,NaN,NaN,NaN,NaN,NaN,NaN
4,syns_in,point,NaN,[864691136333760691],[],00661afd142462468d74167c2857c1d3f64b6870,None,None,1178968.0,798208.0,935680.0,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5272,syns_out,point,NaN,[864691132294257136],[],92fce4b77aa345ad0c5c9e7c2d568b5de9aaa274,None,None,1108608.0,802984.0,908920.0,NaN,NaN,NaN,False,False,False
5273,syns_out,point,NaN,[864691132135519710],[],7aafe74e6a4be5daf64c4f21439ed1788faf0ef0,None,None,1195536.0,759128.0,858120.0,NaN,NaN,NaN,False,False,False
5274,syns_out,point,NaN,[864691132100215248],[],7f250c09e29df2a498236b9177292c0c68ac36f1,None,None,1320728.0,795944.0,954480.0,NaN,NaN,NaN,False,False,False
5275,syns_out,point,NaN,[864691131892380409],[],4199c5a222e64a13976d6628d91eea4807a4fd90,None,None,1306208.0,745784.0,991680.0,NaN,NaN,NaN,False,False,False


In [7]:
from nglui import statebuilder
statebuilder.site_utils.set_default_config(target_site='spelunker')

client.version=1412

AttributeError: module 'nglui.statebuilder.site_utils' has no attribute 'set_default_config'

## Step 2: Load Mesh & Compute/Load HKS
We need the mesh and its HKS features to map our annotations.

In [8]:
print(f"Loading mesh for {target_id}...")
mesh = mm.mesh(seg_id=target_id)

# Run HKS Pipeline (or load if cached - implementing cache logic recommended later)
print("Computing HKS features (this might take a moment)...")
mesh_tuple = (mesh.vertices.astype(np.float32), mesh.faces)
res = condensed_hks_pipeline(
    mesh_tuple, 
    simplify_target_reduction=0.7, 
    distance_threshold=3.0,
    verbose=False
)

# EXTRACT CRITICAL DATA
# 1. The HKS Features per Segment
hks_df = res.condensed_features # Index is usually 0, 1, 2... corresponding to segments

# 2. The simplified mesh vertices (to snap to)
simple_vertices, simple_faces = res.simple_mesh

# 3. Mapping from Vertex Index -> Segment ID
vertex_to_segment = res.simple_labels

print(f"Ready. Mesh has {len(simple_vertices)} vertices and {len(hks_df)} segments.")

Loading mesh for 864691135724233643...
Computing HKS features (this might take a moment)...
Ready. Mesh has 82392 vertices and 2050 segments.


## Step 3: Ingest Annotations & Snap to Mesh
Paste your Neuroglancer JSON state here (or load from file).

In [ ]:
# PASTE JSON HERE
ng_state = {
    # ... paste the full JSON from Neuroglancer here ...
    # dictionary with 'layers': [ ... ]
}

# 1. Extract Points
raw_points = annotation_utils.extract_points_from_ng_state(ng_state, layer_name='annotations')

if len(raw_points) > 0:
    # Neuroglancer points are often in voxel coordinates. 
    # Check if they match mesh units (nm). 
    # If using CAVE/CloudVolume defaults, they might be in 4x4x40nm voxels or similar.
    # adjustments might be needed depending on how the layer was set up.
    # Assuming here points are in NM if view state was consistent, or voxels.
    # TODO: Add explicit unit conversion if needed.
    
    # 2. Snap to Mesh
    dists, v_indices = annotation_utils.snap_points_to_mesh(simple_vertices, raw_points)
    
    print(f"Extracted {len(raw_points)} points. Max distance to mesh: {dists.max():.2f}")
    
    # 3. Map to Segments
    target_segments = annotation_utils.map_indices_to_segments(v_indices, vertex_to_segment)
    
    print(f"Mapped to {len(np.unique(target_segments))} unique segments.")
else:
    print("No points found in state!")

## Step 4: Create ML Dataset (X, y)
We will now fetch the HKS features (X) for these segments and assign label 1 (Spine).

In [ ]:
if len(raw_points) > 0:
    # Create Dataset (Assuming label=1 for these points)
    Xy_df, missing = annotation_utils.create_training_dataset(target_segments, hks_df, label_value=1)
    
    print(f"Created dataset with {len(Xy_df)} samples.")
    if missing:
        print(f"Warning: {len(missing)} segments resulted in missing HKS features.")
        
    # Preview
    print(Xy_df.head())
    
    # Save
    Xy_df.to_csv('spine_training_data.csv', index=False)
    print("Saved to spine_training_data.csv")